# Model Refinement for Bike-Sharing Demand Prediction

After establishing a baseline model, the next step is to improve model performance by incorporating more complex relationships between variables.

The baseline model assumes linear relationships between predictors and the target variable. However, real-world data often exhibits:

- Non-linear relationships
- Interactions between variables
- Overfitting risks with increased complexity

In this notebook, we refine the baseline model by:

- Adding polynomial terms
- Introducing interaction effects
- Applying regularization techniques

This aligns with the second stage of regression modeling as outlined in the lab instructions.

## Import Required Libraries

We use:

- `dplyr`: data manipulation
- `ggplot2`: visualization
- `caret`: model evaluation
- `glmnet`: regularization models

In [ ]:
# Load required libraries

# dplyr for data manipulation
library(dplyr)

# ggplot2 for visualization
library(ggplot2)

# caret for model evaluation
library(caret)

# glmnet for regularization (Ridge, Lasso, Elastic Net)
library(glmnet)

## Load Dataset and Split Data

We load the cleaned dataset and split it into training and testing sets.

In [ ]:
# Load dataset
bike_data <- read_csv("clean_bike_data.csv")

# Set seed for reproducibility
set.seed(123)

# Split dataset (80/20)
train_index <- createDataPartition(bike_data$rented_bike_count, p = 0.8, list = FALSE)

train_data <- bike_data[train_index, ]
test_data <- bike_data[-train_index, ]

## Add Polynomial Terms

Polynomial terms help capture non-linear relationships between variables.

For example:
- Temperature squared may capture diminishing or accelerating effects

In [ ]:
# Add polynomial features

train_data <- train_data %>%
  mutate(
    temperature_sq = temperature^2,
    humidity_sq = humidity^2
  )

test_data <- test_data %>%
  mutate(
    temperature_sq = temperature^2,
    humidity_sq = humidity^2
  )

## Add Interaction Terms

Interaction terms capture combined effects between variables.

For example:
- Temperature × Humidity may influence demand differently than each variable independently

In [ ]:
# Add interaction feature

train_data <- train_data %>%
  mutate(temp_humidity = temperature * humidity)

test_data <- test_data %>%
  mutate(temp_humidity = temperature * humidity)

## Train Enhanced Linear Regression Model

In [ ]:
# Train refined linear model

refined_model <- lm(
  rented_bike_count ~ temperature + humidity + wind_speed +
    temperature_sq + humidity_sq + temp_humidity,
  data = train_data
)

# View model summary
summary(refined_model)

## Generate Predictions

In [ ]:
# Predict on test dataset
predictions_refined <- predict(refined_model, newdata = test_data)

## Evaluate Refined Model

In [ ]:
# RMSE function
rmse <- function(actual, predicted) {
  sqrt(mean((actual - predicted)^2))
}

# Compute RMSE
rmse_refined <- rmse(test_data$rented_bike_count, predictions_refined)

# Compute R-squared
r2_refined <- cor(test_data$rented_bike_count, predictions_refined)^2

# Display metrics
rmse_refined
r2_refined

## Apply Regularization (Ridge, Lasso, Elastic Net)

Regularization helps prevent overfitting by penalizing large coefficients.

We use `glmnet` for:

- Ridge Regression (alpha = 0)
- Lasso Regression (alpha = 1)
- Elastic Net (alpha between 0 and 1)

In [ ]:
# Prepare matrix for glmnet (requires matrix input)

x_train <- model.matrix(rented_bike_count ~ ., train_data)[, -1]
y_train <- train_data$rented_bike_count

x_test <- model.matrix(rented_bike_count ~ ., test_data)[, -1]

# Train Ridge model
ridge_model <- cv.glmnet(x_train, y_train, alpha = 0)

# Train Lasso model
lasso_model <- cv.glmnet(x_train, y_train, alpha = 1)

# Train Elastic Net model
elastic_model <- cv.glmnet(x_train, y_train, alpha = 0.5)

## Predictions with Regularized Models

In [ ]:
# Predict using best lambda

ridge_pred <- predict(ridge_model, s = "lambda.min", newx = x_test)
lasso_pred <- predict(lasso_model, s = "lambda.min", newx = x_test)
elastic_pred <- predict(elastic_model, s = "lambda.min", newx = x_test)

## Compare Model Performance

In [ ]:
# Calculate RMSE for all models

rmse_ridge <- rmse(test_data$rented_bike_count, ridge_pred)
rmse_lasso <- rmse(test_data$rented_bike_count, lasso_pred)
rmse_elastic <- rmse(test_data$rented_bike_count, elastic_pred)

# Display results
rmse_ridge
rmse_lasso
rmse_elastic

## Interpretation

From the refined models:

- Polynomial and interaction terms improve model flexibility
- Regularization reduces overfitting
- Elastic Net often provides a balance between Ridge and Lasso

These improvements enhance prediction accuracy compared to the baseline model.

---

## Author & Acknowledgment

**Author:**  
<span style="color:blue">Deepan Mehta </span>  

**GitHub Profile:**  
https://github.com/deepan-mehta-analytics

This notebook focuses on improving the baseline regression model by introducing polynomial features, interaction terms, and regularization techniques.

The workflow follows <span style="color:blue">IBM Skills Network </span> instructional labs on regression model refinement and performance optimization.

Special acknowledgment is given to:

- <span style="color:blue">Yan Luo </span>  
- <span style="color:blue">Jeff Grossman</span>  

---

## Project Context

This notebook represents the model refinement stage in the end-to-end data science pipeline:

- Data Collection  
- Data Wrangling (ETL)  
- Exploratory Data Analysis (EDA)  
- Baseline Model Development  
- **Model Refinement**  
- Model Evaluation  
- Deployment (R Shiny Dashboard)

---

## Notes

The refined models demonstrate improved predictive capability by capturing non-linear relationships and reducing overfitting through regularization.

---